In [4]:
# ==========================================
# IMPORT DEPENDENCIES & LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import torch
import json
from collections import Counter
import nltk
from nltk.corpus import stopwords
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# ==========================================
# SETUP NLTK STOPWORDS
# ==========================================
# Download NLTK English stopwords for cleaning text keywords during reporting
nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

# ==========================================
# 1. BALANCED SUBSETTING FUNCTION
# ==========================================
def create_balanced_train_test_subset(
    df,
    suburbs,
    topics,
    sample_ratio=0.10,
    suburb_col='suburb',
    topic_col='aspect',
    random_state=42
):
    """
    Extracts a balanced subset from the main DataFrame matching target suburbs and aspects.
    Handles abbreviated CSV aspect strings (e.g., 'Crime & Safe', 'Lifestyle Ame').
    """
    print("Filtering and creating balanced dataset subset...")

    # Normalize target filter lists to lowercase for robust matching
    suburbs_lower = [s.lower() for s in suburbs]
    topics_lower = [t.lower() for t in topics]

    # Flexible filter helper to match CSV aspect naming conventions
    def match_topic(val):
        val_str = str(val).lower()
        return any(t in val_str or any(w in val_str for w in t.split()) for t in topics_lower)

    # Filter main dataframe based on suburbs and aspects
    filtered_df = df[
        df[suburb_col].astype(str).str.lower().isin(suburbs_lower) &
        df[topic_col].astype(str).apply(match_topic)
    ].copy()

    if filtered_df.empty:
        raise ValueError("No matching records found for the specified suburbs and topics in the dataset.")

    # Calculate target samples for stratification across categories
    total_target_samples = int(len(filtered_df) * sample_ratio)
    num_combinations = len(suburbs) * len(topics)
    samples_per_category = max(1, total_target_samples // num_combinations)

    sampled_indices = []

    # Stratify and sample equally across each suburb-topic combination to maintain balance
    for sub in suburbs:
        for top in topics:
            subset = filtered_df[
                (filtered_df[suburb_col].astype(str).str.lower() == sub.lower()) &
                (filtered_df[topic_col].astype(str).str.lower().apply(lambda x: top.lower() in x or any(w in x for w in top.lower().split())))
            ]

            if not subset.empty:
                n_to_sample = min(len(subset), samples_per_category)
                sampled_chunk = subset.sample(n=n_to_sample, random_state=random_state)
                sampled_indices.extend(sampled_chunk.index.tolist())

    # Create the balanced subset for training/testing and reserve the remaining data for JSON reporting
    balanced_subset = df.loc[sampled_indices].reset_index(drop=True)
    report_df = df.drop(sampled_indices).reset_index(drop=True)

    print(f"Balanced subset created with {len(balanced_subset)} rows.")
    print(f"Remaining report dataset has {len(report_df)} rows.")

    return balanced_subset, report_df

In [5]:
# ==========================================
# CUSTOM CALLBACK TO PRINT LOSS & ACCURACY PER EPOCH
# ==========================================
class EpochPrintCallback(TrainerCallback):
    """
    Custom callback to capture and print train loss, train accuracy,
    eval loss, and eval accuracy at the end of every epoch.
    """
    def __init__(self):
        self.last_train_loss = "N/A"

    def on_log(self, args, state, control, logs=None, **kwargs):
        # Capture the training loss whenever logged by the trainer
        if logs is not None and "loss" in logs:
            self.last_train_loss = f"{logs.get('loss'):.4f}"

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        # Triggered during evaluation phases (configured to run per epoch)
        if metrics is not None:
            trainer = kwargs.get("trainer")
            epoch = round(state.epoch, 2) if state.epoch else "N/A"

            # Compute training accuracy across the training dataset
            train_acc = "N/A"
            if trainer is not None and trainer.train_dataset is not None:
                train_preds = trainer.predict(trainer.train_dataset)
                train_acc_val = accuracy_score(train_preds.label_ids, np.argmax(train_preds.predictions, axis=1))
                train_acc = f"{train_acc_val:.4f}"

            # Extract evaluation loss and evaluation accuracy from metrics dictionary
            eval_loss = f"{metrics.get('eval_loss', 'N/A'):.4f}" if isinstance(metrics.get('eval_loss'), (int, float)) else "N/A"
            eval_acc = f"{metrics.get('eval_accuracy', 'N/A'):.4f}" if isinstance(metrics.get('eval_accuracy'), (int, float)) else "N/A"

            print(f"\n[Epoch {epoch}] Train Loss: {self.last_train_loss} | Train Accuracy: {train_acc} | Eval Loss: {eval_loss} | Eval Accuracy: {eval_acc}\n")

# ==========================================
# 2. MODEL TRAINING & EVALUATION (VALIDATION) FUNCTION
# ==========================================
def train_sentiment_model(train_df, text_col='model_text', label_col='sentiment', output_dir="./fine_tuned_model"):
    """
    Fine-tunes the Twitter-RoBERTa model using the categorical 'sentiment' column.
    Splits the input training data into internal train and validation subsets
    to track evaluation metrics during training epochs.
    """
    print("Preparing data for training and internal validation...")

    # Helper mapping function to convert text sentiment strings to integer model labels
    def sentiment_str_to_label(val):
        val_str = str(val).strip().lower()
        if 'negative' in val_str: return 0
        elif 'positive' in val_str: return 2
        else: return 1  # Default to Neutral

    # Drop nulls and assign numerical label column
    clean_df = train_df.dropna(subset=[text_col, label_col]).copy()
    clean_df['label'] = clean_df[label_col].apply(sentiment_str_to_label)

    # Split training data further into an internal training set and a validation (eval) set (80/20 split)
    train_subset, eval_subset = train_test_split(clean_df, test_size=0.2, random_state=42)

    # Format DataFrames into Hugging Face Dataset format
    train_formatted = train_subset[[text_col, 'label']].rename(columns={text_col: 'text'})
    eval_formatted = eval_subset[[text_col, 'label']].rename(columns={text_col: 'text'})

    train_dataset = Dataset.from_pandas(train_formatted)
    eval_dataset = Dataset.from_pandas(eval_formatted)

    # Load pre-trained tokenizer and sequence classification model
    model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    trained_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    # Tokenization function with padding and truncation
    def tokenize_function(examples):
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

    # Define metrics computation function for evaluation during training epochs
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=1)
        acc = accuracy_score(labels, predictions)
        return {"accuracy": acc}

    # Define training arguments configuration with epoch-based evaluation strategy
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",  # Evaluate performance on validation set at the end of each epoch
        logging_strategy="epoch", # Log metrics every epoch
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=5,
        weight_decay=0.07,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none"
    )

    # Initialize the Trainer interface with dataset splits, metrics, and print callback
    trainer = Trainer(
        model=trained_model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        compute_metrics=compute_metrics,
        callbacks=[EpochPrintCallback()]
    )

    print("Starting fine-tuning with per-epoch score printing...")
    trainer.train()
    print("Training and internal evaluation complete!")

    return trainer, trained_model, tokenizer

# ==========================================
# 3. MODEL TESTING FUNCTION
# ==========================================
def test_sentiment_model(trainer, tokenizer, test_df, text_col='model_text', label_col='sentiment'):
    """
    Evaluates the final fine-tuned model against a completely unseen test dataset
    using the Trainer API to prevent device placement conflicts.
    """
    print("Running final testing on unseen test dataset...")

    # Helper mapping function to convert test labels into matching integers
    def sentiment_str_to_label(val):
        val_str = str(val).strip().lower()
        if 'negative' in val_str: return 0
        elif 'positive' in val_str: return 2
        else: return 1

    # Clean and isolate ground truth labels
    test_df = test_df.dropna(subset=[text_col, label_col]).copy()
    true_labels = test_df[label_col].apply(sentiment_str_to_label).tolist()

    # Prepare test data layout for tokenizer
    formatted_df = test_df[[text_col]].rename(columns={text_col: 'text'})
    test_dataset = Dataset.from_pandas(formatted_df)

    def tokenize_function(examples):
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

    tokenized_test = test_dataset.map(tokenize_function, batched=True)

    # Run prediction through Trainer API (automatically manages device batch placement)
    predictions_output = trainer.predict(tokenized_test)
    logits = predictions_output.predictions
    predicted_labels = np.argmax(logits, axis=1).tolist()

    # Calculate final performance metrics on test split
    accuracy = accuracy_score(true_labels, predicted_labels)
    report = classification_report(true_labels, predicted_labels, target_names=['Negative', 'Neutral', 'Positive'], zero_division=0)

    print(f"\n--- FINAL TEST EVALUATION RESULTS ---")
    print(f"Test Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(report)

    return accuracy, predicted_labels


# ==========================================
# MAIN EXECUTION PIPELINE FLOW
# ==========================================
if __name__ == "__main__":
    # Define scope configurations for Melbourne suburbs and core urban aspects
    target_suburbs = [
        "Collingwood", "Oakleigh", "Nunawading", "Eaglemont",
        "Richmond", "Carlton", "Vermont", "Dandenong", "Officer", "Clayton"
    ]

    target_topics = [
        "lifestyle",
        "education",
        "crime",
        "affordability"
    ]

    # Step 1: Load raw dataset from CSV
    df = pd.read_csv('melbourne_suburb_training_subset.csv')

    # Step 2: Extract a stratified 10% balanced subset and reserve the 90% report data
    balanced_10_pct, report_data = create_balanced_train_test_subset(
        df=df,
        suburbs=target_suburbs,
        topics=target_topics,
        sample_ratio=0.30,
        suburb_col='suburb',
        topic_col='aspect'
    )

    # Step 3: Split the balanced subset further into Training (80%) and Testing (20%) splits
    train_data, test_data = train_test_split(balanced_10_pct, test_size=0.2, random_state=42)

    # Step 4: Fine-tune the model using the categorical 'sentiment' label column
    trainer, trained_model, tokenizer = train_sentiment_model(train_data, label_col='sentiment')

    # Step 5: Evaluate model performance on the test split using the Trainer API
    accuracy, predictions = test_sentiment_model(trainer, tokenizer, test_data, label_col='sentiment')

Filtering and creating balanced dataset subset...
Balanced subset created with 276 rows.
Remaining report dataset has 1077 rows.
Preparing data for training and internal validation...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/176 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

...


[Epoch 5.0] Train Loss: 0.1541 | Train Accuracy: N/A | Eval Loss: 1.6198 | Eval Accuracy: 0.5682



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training and internal evaluation complete!
Running final testing on unseen test dataset...


Map:   0%|          | 0/56 [00:00<?, ? examples/s]


--- FINAL TEST EVALUATION RESULTS ---
Test Accuracy: 0.7321

Classification Report:
              precision    recall  f1-score   support

    Negative       0.57      0.80      0.67        15
     Neutral       0.88      0.64      0.74        22
    Positive       0.79      0.79      0.79        19

    accuracy                           0.73        56
   macro avg       0.75      0.74      0.73        56
weighted avg       0.76      0.73      0.74        56



In [6]:
import json
import numpy as np
import torch
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
try:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    nltk_stopwords = set(stopwords.words('english'))
except Exception:
    nltk_stopwords = set()

# Combine NLTK stopwords with your custom dataset stopwords
custom_stopwords = list(nltk_stopwords) + [
    # filler verbs
    "get", "gets", "got", "getting", "go", "goes", "gone", "going",
    "come", "came", "comes", "said", "saying", "say", "told", "telling", "tell",
    "asked", "ask", "asking", "talk", "talked", "talking", "talkin",
    "make", "making", "give", "gives", "take", "takes", "taking", "taken",
    "tried", "try", "wanted", "want", "saw", "see", "watched", "watching",
    "heard", "read", "run", "running", "called", "call", "reply", "comment",
    "post", "search", "add", "adding", "click", "provide", "explain", "send",
    "guess", "wonder", "wondering", "imagine", "consider", "assume", "assumed",
    "mind", "pick", "appears", "happened", "happens", "started", "finished",
    "entered", "headed", "heading", "left", "wish", "might", "may", "know",
    "think", "thought", "feel", "feels", "seems", "let", "put", "use", "using",
    "used", "bought", "carried", "brought", "forgot", "forgets", "join",
    "show", "shows", "include", "including",

    # generic nouns
    "thing", "things", "stuff", "lot", "lots", "bunch", "bit", "bits", "kind",
    "sort", "way", "times", "time", "word", "name", "names", "guy", "guys",
    "man", "dude", "fella", "bros", "everyone", "everything", "people", "one",
    "ones", "story", "question", "idea", "case", "example", "examples",
    "reason", "number", "moment", "day", "days", "month", "weeks", "weekend",
    "hour", "minutes", "second", "half", "part", "terms", "context", "info",
    "information", "feelings",

    # hedges, adverbs and connectors
    "actually", "basically", "honestly", "obviously", "apparently",
    "definitely", "certainly", "clearly", "exactly", "especially",
    "specifically", "simply", "frankly", "totally", "completely", "fully",
    "absolutely", "almost", "likely", "probably", "usually", "currently",
    "recently", "eventually", "finally", "essentially", "largely", "properly",
    "anyway", "else", "instead", "rather", "yet", "since", "though",
    "although", "across", "along", "upon", "via", "within", "whether",
    "except", "enough", "plus", "far", "another", "others", "given", "based",
    "real", "actual", "sure", "specific", "entire", "unlike", "anywhere",
    "somewhere", "earlier", "next",

    # internet slang and artefacts
    "haha", "lol", "btw", "tbh", "idk", "fyi", "yep", "nah", "hey", "awww",
    "gonna", "gotta", "ive", "thry", "belive", "atm", "stfu", "gaf", "goto",
    "remindmebot", "bot", "forum", "link", "online", "facebook", "instagram",
    "google", "ads", "internet", "web", "webs", "email", "text", "episode",
    "video", "podcasts", "data", "analysis", "code", "spreadsheet", "charts",
    "tool", "filter", "filters", "auto", "automated", "automatic", "edit",

    # usernames and personal names
    "kenny", "smith", "andrew", "cheryl", "compton", "trev", "luke", "sayers",
    "wilson", "william", "webster", "wang", "leehom", "amara", "trenery",
    "squires", "sutherland", "marquis", "raskals", "babybumpsandbeyondau",
    "chinesepod", "wick", "jesus",

    # place names (delete this block if location matters to your analysis)
    "collingwood", "oakleigh", "nunawading", "eaglemont", "richmond",
    "carlton", "vermont", "dandenong", "officer", "clayton",
    "brunswick", "footscray", "greensborough", "eltham", "docklands",
    "bourke", "frankston", "cranbourne", "moonee", "ponds", "ringwood",
    "sorrento", "warrigal", "bayswater", "croydon", "heidelberg", "watsonia",
    "banyule", "rosanna", "macleod", "darebin", "glen", "hawksburn",
    "elsternwick", "albans", "belgrave", "heathmont", "eastland", "preston",
    "hawthorn", "vale", "lygon", "fitzroy", "yarra", "viewbank", "wantirna",
    "werribee", "braybrook", "altona", "hallam", "hampton", "sunshine",
    "prahran", "clyde", "monash", "chisholm", "sydney", "brisbane",
    "victoria", "victorians", "melb", "melbourne", "australia", "australian",
    "aussie", "aussies", "byron", "grampians", "bacchus", "gold", "coast",
    "europe", "nyc", "vegas", "leura", "heathmore", "suburb", "suburbs",
    "area", "areas",

    # profanity used as filler (words like "shithole" and "garbage" are kept
    # on purpose because they carry negative sentiment)
    "fucking", "fuck", "cunts", "cunt", "arse", "arsehole", "bloody",
    "actually", "also", "already", "although", "another",
    "anyway", "apparently", "around", "basically", "believe",
    "btw", "certainly", "clearly", "completely", "definitely",
    "else", "enough", "especially", "exactly", "fairly",
    "fully", "generally", "guess", "haha", "honestly",
    "however", "imagine", "instead", "just", "kind",
    "literally", "maybe", "might", "much", "nah",
    "obviously", "okay", "ones", "perhaps", "pretty",
    "probably", "quite", "rather", "really", "said",
    "saying", "seems", "simply", "someone", "something",
    "sometimes", "somewhere", "sort", "still", "stuff",
    "sure", "things", "think", "though", "totally",
    "true", "usually", "whatever", "yeah", "yep",
    "yet", "guys", "guy", "dude", "fyi", "lol"
]

custom_stopwords += [
    "like", "would", "could", "well", "thanks", "thank", "looking", "went", "took",
    "url", "need", "hope", "better", "good", "best", "great", "nice", "first", "ever",
    "found", "always", "never", "back", "years", "mate", "per", "help", "check",
    "posting", "downvoted", "sounds", "seen", "stop", "cool", "awesome", "worry", "follow",
    "coburg", "glenroy", "south"
]

custom_stopwords += [
    "even", "dont", "etc", "made", "must", "either", "anything", "please", "needs",
    "start", "reddit", "keep", "new", "happy", "year", "past", "end", "main", "front",
    "checked", "covered", "average", "understanding", "face", "hell", "shit", "west",
    "east", "perceived", "signs", "song", "wow", "life", "home", "north", "living", "reduce", "problem"
]

# ==========================================
# AUTOMATED GLOBAL FREQUENCY FILTER (Strategy 2)
# ==========================================
def filter_ubiquitous_keywords(suburb_summaries, max_topic_occurrences=3, final_n=5):  # FIX: final_n
    """
    Pass 2: Scans all generated keywords across the dataset and removes
    any terms that appear across more than `max_topic_occurrences` topics/suburbs
    to automatically eliminate common noise words.
    """
    keyword_counter = Counter()

    # Count how many times each keyword appears across all topics
    for item in suburb_summaries:
        suburb_item = item.get("suburb_summary", item)
        for topic_data in suburb_item.get("topics", {}).values():
            for kw in topic_data.get("keywords", []):
                keyword_counter[kw] += 1

    # Identify overly common keywords exceeding the threshold
    ubiquitous_words = {kw for kw, count in keyword_counter.items() if count > max_topic_occurrences}

    # Filter them out of topics, strengths, and concerns
    for item in suburb_summaries:
        suburb_item = item.get("suburb_summary", item)
        for topic_name, topic_data in suburb_item.get("topics", {}).items():
            if not topic_data:
                continue
            # FIX: trim to final_n after filtering
            filtered_kws = [kw for kw in topic_data.get("keywords", []) if kw not in ubiquitous_words][:final_n]
            if not filtered_kws:
                filtered_kws = [topic_name.lower()]  # Fallback if empty
            topic_data["keywords"] = filtered_kws

        # Re-sync top strengths and concerns lists with the newly filtered keywords
        suburb_item["top_strengths"] = [
            {**t, "keywords": [kw for kw in t["keywords"] if kw not in ubiquitous_words][:final_n] or [t["topic"].lower()]}  # FIX
            for t in suburb_item.get("top_strengths", [])
        ]
        suburb_item["top_concerns"] = [
            {**t, "keywords": [kw for kw in t["keywords"] if kw not in ubiquitous_words][:final_n] or [t["topic"].lower()]}  # FIX
            for t in suburb_item.get("top_concerns", [])
        ]

    return suburb_summaries


# ==========================================
# 4. JSON REPORT GENERATION FUNCTION
# ==========================================
def generate_suburb_sentiment_summary(
    trained_model,
    tokenizer,
    df,
    suburbs,
    topics,
    topics_2,
    stopwords_list=None,
    text_col='model_text',
    topic_col='aspect',
    source_col='source_type'
):
    trained_model.eval()
    device = next(trained_model.parameters()).device
    suburb_summaries = []

    stopwords = list(stopwords_list) if stopwords_list else custom_stopwords

    for sub in suburbs:
        sub_df = df[df['suburb'].astype(str).str.lower() == sub.lower()]
        if sub_df.empty:
            continue

        if source_col in sub_df.columns:
            total_posts = len(sub_df[sub_df[source_col].astype(str).str.lower() == 'post'])
            total_comments = len(sub_df[sub_df[source_col].astype(str).str.lower() == 'comment'])
        else:
            total_posts = len(sub_df)
            total_comments = 0

        predictions = []
        continuous_scores = []
        texts = sub_df[text_col].astype(str).tolist()

        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = trained_model(**inputs)
                logits = outputs.logits
                pred_class = torch.argmax(logits, dim=1).item()
                predictions.append(pred_class)
                probs = torch.softmax(logits, dim=1)[0]
                cont_score = probs[2].item() - probs[0].item()
                continuous_scores.append(cont_score)

        if not predictions:
            continue

        overall_score = float(np.mean(continuous_scores))

        if overall_score > 0.05:
            overall_label = "Positive"
        elif overall_score < -0.05:
            overall_label = "Negative"
        else:
            overall_label = "Neutral"

        pos_count = predictions.count(2)
        neu_count = predictions.count(1)
        neg_count = predictions.count(0)
        total_count = len(predictions)

        pos_pct = round((pos_count / total_count) * 100) if total_count > 0 else 0
        neu_pct = round((neu_count / total_count) * 100) if total_count > 0 else 0
        neg_pct = round((neg_count / total_count) * 100) if total_count > 0 else 0

        top_strengths = []
        top_concerns = []
        topics_summary_dict = {}

        for i, topic in enumerate(topics):
            topic_name = topics_2[i]

            # FIX: drop "&" and other short tokens, otherwise "&" matches every
            # aspect like "Crime & Safe" / "Education & Employment" and all
            # topics receive the same posts.
            topic_words = [w for w in topic.lower().split() if len(w) > 3]
            topic_df = sub_df[sub_df[topic_col].astype(str).str.lower().apply(
                lambda x: topic.lower() in x or any(w in x for w in topic_words)
            )]

            if topic_df.empty:
                topics_summary_dict[topic_name] = {
                    "topic": topic_name,
                    "average_score": 0.0,
                    "label": "Neutral",
                    "post_count": 0,
                    "keywords": []
                }
                continue

            topic_texts = topic_df[text_col].astype(str).tolist()
            topic_scores = []

            for t in topic_texts:
                inputs = tokenizer(t, return_tensors="pt", truncation=True, max_length=128)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                with torch.no_grad():
                    outs = trained_model(**inputs)
                    p = torch.softmax(outs.logits, dim=1)[0]
                    topic_scores.append(p[2].item() - p[0].item())

            avg_topic_score = float(np.mean(topic_scores)) if topic_scores else 0.0

            if avg_topic_score > 0.05:
                topic_label = "Positive"
            elif avg_topic_score < -0.05:
                topic_label = "Negative"
            else:
                topic_label = "Neutral"

            # Extract keywords using TF-IDF Vectorizer with stopwords
            common_keywords = []
            if topic_texts:
                try:
                    vectorizer = TfidfVectorizer(
                        stop_words=stopwords if stopwords else None,
                        max_features=100,
                        min_df=2,             #FIX: word must appear in 2+ posts
                        token_pattern=r'(?u)\b[A-Za-z]{3,}\b'
                    )
                    X_tfidf = vectorizer.fit_transform(topic_texts)
                    sums = X_tfidf.sum(axis=0).A1
                    sorted_inds = np.argsort(sums)[::-1]
                    feature_names = vectorizer.get_feature_names_out()
                    # FIX: keep 15 candidates so some survive the global filter
                    common_keywords = [feature_names[idx] for idx in sorted_inds[:15]]
                except Exception:
                    common_keywords = []

            if not common_keywords:
                words = " ".join(topic_texts).lower().split()
                filtered_words = [w for w in words if w.isalpha() and w not in stopwords and len(w) > 2]
                # FIX: was most_common(100)
                common_keywords = [word for word, count in Counter(filtered_words).most_common(15)]
                if not common_keywords:
                    common_keywords = [topic.lower()]

            topic_info = {
                "topic": topic_name,
                "average_score": round(avg_topic_score, 2),
                "label": topic_label,
                "post_count": len(topic_df),
                "keywords": common_keywords
            }

            topics_summary_dict[topic_name] = topic_info

            if avg_topic_score >= 0.05:
                top_strengths.append(topic_info)
            elif avg_topic_score <= -0.05:
                top_concerns.append(topic_info)

        suburb_summary = {
            "suburb": sub,
            "overall_sentiment_score": round(overall_score, 2),
            "sentiment_label": overall_label,
            "post_count": total_posts,
            "comment_count": total_comments,
            "sentiment_breakdown": {
                "positive_pct": pos_pct,
                "neutral_pct": neu_pct,
                "negative_pct": neg_pct
            },
            "top_strengths": top_strengths,
            "top_concerns": top_concerns,
            "topics": {
                "Lifestyle & Amenities": topics_summary_dict.get("Lifestyle & Amenities", {}),
                "Education & Employment": topics_summary_dict.get("Education & Employment", {}),
                "Crime & Safety": topics_summary_dict.get("Crime & Safety", {}),
                "Affordability": topics_summary_dict.get("Affordability", {})
            }
        }
        suburb_summaries.append(suburb_summary)

    # FIX: only drop words used in more than half of all suburb-topic slots
    # (was 4 of 40, which removed the most relevant words like "schools", "rent").
    max_occ = (len(suburbs) * len(topics)) // 2
    suburb_summaries = filter_ubiquitous_keywords(suburb_summaries, max_topic_occurrences=max_occ, final_n=3)

    return {
        "last_updated": "2026-10-19",
        "suburb_summary": suburb_summaries
    }

In [7]:
target_topics_2 = [
    "Lifestyle & Amenities",
    "Education & Employment",
    "Crime & Safety",
    "Affordability"
]
# 6. Generate final summaries using the broader reporting data with custom stopwords
summary_output = generate_suburb_sentiment_summary(
    trained_model=trained_model,
    tokenizer=tokenizer,
    df=report_data,
    suburbs=target_suburbs,
    topics=target_topics_2,
    topics_2=target_topics_2,
    stopwords_list=custom_stopwords  # Pass your NLTK + custom stopword list here
)

# 7. Save the output directly as a JSON file
output_filename = "suburb_sentiment_summary.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(summary_output, f, indent=4, ensure_ascii=False)

print(f"\nSuccessfully generated and saved summary JSON to '{output_filename}'!")


Successfully generated and saved summary JSON to 'suburb_sentiment_summary.json'!


In [3]:

# ==========================================
# MAIN EXECUTION PIPELINE FLOW (DIRECT TESTING - NO TRAINING)
# ==========================================
def test_sentiment_model(trainer, tokenizer, test_df, text_col='model_text', label_col='sentiment'):
    """
    Evaluates the final fine-tuned model against a completely unseen test dataset
    using the Trainer API to prevent device placement conflicts.
    """
    print("Running final testing on unseen test dataset...")

    # Helper mapping function to convert test labels into matching integers
    def sentiment_str_to_label(val):
        val_str = str(val).strip().lower()
        if 'negative' in val_str: return 0
        elif 'positive' in val_str: return 2
        else: return 1

    # Clean and isolate ground truth labels
    test_df = test_df.dropna(subset=[text_col, label_col]).copy()
    true_labels = test_df[label_col].apply(sentiment_str_to_label).tolist()

    # Prepare test data layout for tokenizer
    formatted_df = test_df[[text_col]].rename(columns={text_col: 'text'})
    test_dataset = Dataset.from_pandas(formatted_df)

    def tokenize_function(examples):
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

    tokenized_test = test_dataset.map(tokenize_function, batched=True)

    # Run prediction through Trainer API (automatically manages device batch placement)
    predictions_output = trainer.predict(tokenized_test)
    logits = predictions_output.predictions
    predicted_labels = np.argmax(logits, axis=1).tolist()

    # Calculate final performance metrics on test split
    accuracy = accuracy_score(true_labels, predicted_labels)
    report = classification_report(true_labels, predicted_labels, target_names=['Negative', 'Neutral', 'Positive'], zero_division=0)

    print(f"\n--- FINAL TEST EVALUATION RESULTS ---")
    print(f"Test Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(report)

    return accuracy, predicted_labels
if __name__ == "__main__":
    # Define scope configurations for Melbourne suburbs and core urban aspects
    target_suburbs = [
        "Collingwood", "Oakleigh", "Nunawading", "Eaglemont",
        "Richmond", "Carlton", "Vermont", "Dandenong", "Officer", "Clayton"
    ]

    target_topics = [
        "lifestyle",
        "education",
        "crime",
        "affordability"
    ]

    # Step 1: Load raw dataset from CSV
    df = pd.read_csv('melbourne_suburb_training_subset.csv')

    # Step 2: Extract a stratified balanced subset and reserve report data
    balanced_10_pct, report_data = create_balanced_train_test_subset(
        df=df,
        suburbs=target_suburbs,
        topics=target_topics,
        sample_ratio=0.30,
        suburb_col='suburb',
        topic_col='aspect'
    )

    # Step 3: Split the balanced subset into Training and Testing splits (we use test_data for direct evaluation)
    train_data, test_data = train_test_split(balanced_10_pct, test_size=0.2, random_state=42)

    # Step 4: Load the pre-trained model and tokenizer directly WITHOUT fine-tuning
    print("Loading pre-trained model directly without fine-tuning...")
    model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    # Initialize a lightweight Trainer instance solely for running .predict() without device/batch issues
    eval_training_args = TrainingArguments(
        output_dir="./temp_eval_dir",
        per_device_eval_batch_size=8,
        report_to="none"
    )

    eval_trainer = Trainer(
        model=base_model,
        args=eval_training_args
    )

    # Step 5: Evaluate the out-of-the-box pre-trained model on the test split
    accuracy, predictions = test_sentiment_model(eval_trainer, tokenizer, test_data, label_col='sentiment')

Filtering and creating balanced dataset subset...
Balanced subset created with 276 rows.
Remaining report dataset has 1077 rows.
Loading pre-trained model directly without fine-tuning...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running final testing on unseen test dataset...


Map:   0%|          | 0/56 [00:00<?, ? examples/s]


--- FINAL TEST EVALUATION RESULTS ---
Test Accuracy: 0.5714

Classification Report:
              precision    recall  f1-score   support

    Negative       0.50      0.73      0.59        15
     Neutral       0.54      0.64      0.58        22
    Positive       0.88      0.37      0.52        19

    accuracy                           0.57        56
   macro avg       0.64      0.58      0.57        56
weighted avg       0.64      0.57      0.56        56

